# Vibepixel Avatar Pipeline

This notebook mirrors the Colab flow for spreadsheet imports, preview, and export. The first import cell tries the local `avatar_pipeline.py` first and, if it is missing, downloads it automatically from the public repository so you do not need to copy files by hand. Keep RGB values quoted as a single CSV field, for example `"255,204,0"`.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import urllib.request

AVATAR_PIPELINE_URL = "https://raw.githubusercontent.com/HaroldSthid/VibePixel-AcademyBabyStepsTech/main/artifacts/colab/avatar_pipeline.py"
MODULE_PATH = Path("avatar_pipeline.py")

def _load_avatar_pipeline():
    try:
        from avatar_pipeline import export_avatar, load_avatar_frames, preview_avatar
        return export_avatar, load_avatar_frames, preview_avatar
    except ImportError:
        if not MODULE_PATH.exists():
            try:
                urllib.request.urlretrieve(AVATAR_PIPELINE_URL, MODULE_PATH)
            except Exception as download_exc:
                raise ImportError(
                    "No pudimos cargar avatar_pipeline.py automáticamente. Verificá que tengas conexión a internet y volvé a ejecutar la celda."
                ) from download_exc

        try:
            spec = importlib.util.spec_from_file_location("avatar_pipeline", MODULE_PATH)
            if spec is None or spec.loader is None:
                raise ImportError("No se pudo preparar la carga de avatar_pipeline.py.")
            module = importlib.util.module_from_spec(spec)
            sys.modules["avatar_pipeline"] = module
            spec.loader.exec_module(module)
            return module.export_avatar, module.load_avatar_frames, module.preview_avatar
        except Exception as load_exc:
            raise ImportError(
                "No pudimos cargar avatar_pipeline.py automáticamente. Verificá que el notebook tenga acceso a internet y volvé a ejecutar la celda."
            ) from load_exc

export_avatar, load_avatar_frames, preview_avatar = _load_avatar_pipeline()


In [ ]:
source_csv = Path("avatar-16x16.csv")  # Replace with your exported spreadsheet CSV.
frames = load_avatar_frames(source_csv)
len(frames), [frame_id for frame_id, _ in frames]


In [ ]:
preview = preview_avatar(frames, scale=16)
preview


In [ ]:
result = export_avatar(frames, output_dir=Path("exports"), stem="avatar")
result


## Single-frame fallback

If the export contains only one frame, the pipeline writes `avatar_preview.png` instead of an animated GIF. Use that still image for the next workshop step. Run `validate_avatar_pipeline.py` to verify the import, preview, and export contract before sharing the artifact. The import cell already handles the download fallback automatically.
